# SAM3 Faz 4 — Eğitim

Bu notebook şunları yapar:
1. Google Drive'dan veri setini bağlar
2. Modeli yükler ve LoRA uygular
3. Küçük bir alt kümeyle test eğitimi yapar
4. Eğitilmiş modeli Drive'a kaydeder

**ÖNEMLİ:** Çalıştırmadan önce `Runtime > Change runtime type > T4 GPU` seç!

## Adım 0 — GPU Kontrolü

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU aktif: {torch.cuda.get_device_name(0)}")
    print(f"GPU bellek: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("UYARI: GPU bulunamadı! Runtime > Change runtime type > T4 GPU seç.")

## Adım 1 — Paket Kurulumu

In [ ]:
!pip install -q transformers peft accelerate

## Adım 2 — HuggingFace Login

Sol menüde 🔑 Secrets'a HF_TOKEN ekle (Faz 3'te yaptığın gibi).

In [ ]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
print("HuggingFace girişi başarılı!")

## Adım 3 — Google Drive Bağlantısı

Veri seti ve checkpointler Drive'da saklanacak.
Çalıştırınca izin isteyecek, onayla.

In [ ]:
from google.colab import drive

# Drive'ı bağla — izin penceresi açılacak, onayla
# Hata alırsan: Runtime > Disconnect and delete runtime → sayfayı yenile → tekrar çalıştır
drive.mount('/content/drive', force_remount=True)

# Veri seti yolları (Drive'daki sam3/ klasörüne göre)
DRIVE_ROOT    = "/content/drive/MyDrive/sam3"
TRAIN_IMAGES  = f"{DRIVE_ROOT}/dacl10k/images/train"
TRAIN_ANNOTS  = f"{DRIVE_ROOT}/dacl10k/annotations/train"
VAL_IMAGES    = f"{DRIVE_ROOT}/dacl10k/images/validation"
VAL_ANNOTS    = f"{DRIVE_ROOT}/dacl10k/annotations/validation"
CHECKPOINT_DIR = f"{DRIVE_ROOT}/checkpoints"

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Kaç görsel var?
train_count = len([f for f in os.listdir(TRAIN_IMAGES) if f.endswith('.jpg')])
val_count   = len([f for f in os.listdir(VAL_IMAGES)   if f.endswith('.jpg')])
print(f"Eğitim görseli  : {train_count}")
print(f"Validation görseli: {val_count}")
print("Drive bağlandı!")

## Adım 4 — Model ve LoRA Yükleme

Faz 3'te test ettiğimiz adımların aynısı.

In [ ]:
from transformers import Sam3Model, Sam3Processor
from peft import LoraConfig, get_peft_model

MODEL_NAME = "facebook/sam3"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Processor yükleniyor...")
processor = Sam3Processor.from_pretrained(MODEL_NAME)

print("Model yükleniyor (birkaç dakika sürebilir)...")
# torch_dtype=torch.float16 → model yarı hassasiyette yüklenir, bellek kullanımı ~yarıya düşer
model = Sam3Model.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
model = model.to(DEVICE)

print("LoRA uygulanıyor...")
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"],
)
model = get_peft_model(model, lora_config)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nEğitilebilir parametreler: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
print(f"Model hazır. Cihaz: {DEVICE}")

## Adım 5 — Dataset Hazırlama

Önce **küçük bir alt kümeyle** test edelim (100 görsel).
Her şey yolunda giderse tam veri setiyle devam ederiz.

In [ ]:
import json
import numpy as np
from PIL import Image, ImageDraw
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# 19 sınıf listesi
DACL10K_CLASSES = [
    "Crack", "ACrack", "Spalling", "Efflorescence", "ExposedRebars",
    "Cavity", "Restformwork", "Rockpocket", "Hollowareas",
    "Rust", "Weathering", "Graffiti", "Wetspot",
    "Bearing", "Drainage", "EJoint", "JTape", "PEquipment", "WConccor",
]
NUM_CLASSES = len(DACL10K_CLASSES)

# SAM3 mask decoder çıktı boyutu — model her zaman 288x288 mask üretir
MASK_OUTPUT_SIZE = 288

# Metin ipucu — modele "ne arayacağını" söylüyoruz
TEXT_PROMPT = "damage"

# Metin tokenlarını bir kez önceden hesapla (her örnek için aynı metin)
# processor.image_processor → görseli işler
# processor.tokenizer       → metni işler
_text_inputs = processor.tokenizer(
    TEXT_PROMPT,
    return_tensors="pt",
    padding=True,
    truncation=True,
)
TEXT_INPUT_IDS      = _text_inputs["input_ids"].squeeze(0)       # (seq_len,)
TEXT_ATTENTION_MASK = _text_inputs["attention_mask"].squeeze(0)  # (seq_len,)

print(f"Text prompt    : '{TEXT_PROMPT}'")
print(f"Token sayısı   : {TEXT_INPUT_IDS.shape[0]}")


def _polygon_to_mask(points, height, width):
    """Polygon koordinatlarından binary mask oluşturur."""
    channel = Image.new("L", (width, height), 0)
    drawer = ImageDraw.Draw(channel)
    point_list = [(int(p[0]), int(p[1])) for p in points]
    if len(point_list) >= 3:
        drawer.polygon(point_list, fill=1)
    return np.array(channel, dtype=np.uint8)


def _json_to_mask(json_path, height, width):
    """JSON annotation dosyasından 19 kanallı mask oluşturur."""
    with open(json_path, "r", encoding="utf-8") as f:
        annotation = json.load(f)
    mask = np.zeros((height, width, NUM_CLASSES), dtype=np.uint8)
    for shape in annotation.get("shapes", []):
        class_name = shape.get("label", "")
        if class_name not in DACL10K_CLASSES:
            continue
        class_idx = DACL10K_CLASSES.index(class_name)
        points = shape.get("points", [])
        if len(points) < 3:
            continue
        polygon_mask = _polygon_to_mask(points, height, width)
        mask[:, :, class_idx] = np.maximum(mask[:, :, class_idx], polygon_mask)
    return mask


class DACL10KDataset(Dataset):
    """DACL10K veri seti — görseller ve maskler."""

    def __init__(self, images_dir, annotations_dir, processor, max_samples=None):
        self.images_dir = images_dir
        self.annotations_dir = annotations_dir
        self.processor = processor

        dosyalar = sorted([
            os.path.splitext(f)[0]
            for f in os.listdir(images_dir)
            if f.lower().endswith(".jpg")
        ])

        if max_samples:
            dosyalar = dosyalar[:max_samples]

        self.file_list = dosyalar
        print(f"Dataset hazır: {len(self.file_list)} görsel")

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        base_name = self.file_list[idx]

        # --- Görseli oku ---
        image_path = os.path.join(self.images_dir, base_name + ".jpg")
        image = Image.open(image_path).convert("RGB")
        width, height = image.size

        # --- Mask oluştur ---
        json_path = os.path.join(self.annotations_dir, base_name + ".json")
        if os.path.exists(json_path):
            mask = _json_to_mask(json_path, height, width)
        else:
            mask = np.zeros((height, width, NUM_CLASSES), dtype=np.uint8)

        # Tüm sınıfları birleştir → tek binary mask (hasar var=1, yok=0)
        binary_mask = mask.any(axis=2).astype(np.float32)

        # Mask'ı 288x288'e küçült (model çıktısıyla aynı boyut)
        mask_pil = Image.fromarray((binary_mask * 255).astype(np.uint8))
        mask_resized = mask_pil.resize(
            (MASK_OUTPUT_SIZE, MASK_OUTPUT_SIZE), Image.NEAREST
        )
        mask_resized = np.array(mask_resized, dtype=np.float32) / 255.0

        # --- Görseli işle (metin AYRI işleniyor) ---
        # image_processor sadece görseli alır, metin ona verilmez
        image_inputs = self.processor.image_processor(
            images=image,
            return_tensors="pt",
        )
        pixel_values = image_inputs["pixel_values"].squeeze(0)  # (3, 1008, 1008)

        return {
            "pixel_values"     : pixel_values,
            "input_ids"        : TEXT_INPUT_IDS.clone(),
            "attention_mask"   : TEXT_ATTENTION_MASK.clone(),
            "ground_truth_mask": torch.tensor(mask_resized),
        }


# --- Dataset oluştur ---
TEST_SAMPLE_SIZE = 100

train_dataset = DACL10KDataset(
    images_dir=TRAIN_IMAGES,
    annotations_dir=TRAIN_ANNOTS,
    processor=processor,
    max_samples=TEST_SAMPLE_SIZE,
)

val_dataset = DACL10KDataset(
    images_dir=VAL_IMAGES,
    annotations_dir=VAL_ANNOTS,
    processor=processor,
    max_samples=20,
)

# Batch size=1 — SAM3 büyük model, T4 GPU'da 2 yetmiyor
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=1, shuffle=False)

print(f"\nMask boyutu: {MASK_OUTPUT_SIZE}x{MASK_OUTPUT_SIZE}")
print(f"Eğitim batch sayısı    : {len(train_loader)}")
print(f"Validation batch sayısı: {len(val_loader)}")

## Adım 6 — Tek Batch Test

Eğitime başlamadan önce **tek bir batch** üzerinde forward pass çalışıyor mu diye kontrol ediyoruz.
Bu adım hata varsa erken yakalar.

In [ ]:
# Tek batch al ve TAM forward pass testi yap
batch = next(iter(train_loader))

print("Batch içeriği:")
for key, value in batch.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: {value.shape}")

# Tüm tensor'ları GPU'ya taşı
pixel_values   = batch["pixel_values"].to(DEVICE)
input_ids      = batch["input_ids"].to(DEVICE)
attention_mask = batch["attention_mask"].to(DEVICE)
gt_mask        = batch["ground_truth_mask"].to(DEVICE)

# TAM forward pass: Image + Text → Encoder → DETR → Mask Decoder
with torch.no_grad():
    outputs = model(
        pixel_values=pixel_values,
        input_ids=input_ids,
        attention_mask=attention_mask,
    )

print(f"\n--- Model Çıktıları ---")
print(f"pred_masks   : {outputs.pred_masks.shape}")     # (B, 100, 288, 288) — her nesne için ayrı mask
print(f"pred_boxes   : {outputs.pred_boxes.shape}")      # (B, 100, 4)       — nesne kutuları
print(f"pred_logits  : {outputs.pred_logits.shape}")     # (B, 100)          — güven skorları
print(f"semantic_seg : {outputs.semantic_seg.shape}")     # (B, 1, 288, 288)  — piksel piksel segmentation
print(f"ground_truth : {gt_mask.shape}")                  # (B, 288, 288)     — hedef mask

print(f"\nsemantic_seg değer aralığı: [{outputs.semantic_seg.min():.2f}, {outputs.semantic_seg.max():.2f}]")
print("\nTam forward pass başarılı! Eğitime geçilebilir.")

## Adım 7 — Eğitim

**Pipeline (tam akış):**
```
Image + "damage" metni
  → Vision Encoder (görseli analiz eder)
  → Text Encoder (metni analiz eder)
  → DETR Encoder (görsel + metin birleşir)
  → DETR Decoder (nesneleri bulur)
  → Mask Decoder (piksel piksel mask üretir)
```

**Ne yapıyoruz?**
- Her görsel + "damage" metni modele veriliyor
- Model tam pipeline'dan geçip `semantic_seg` çıktısı üretiyor (288×288 piksel piksel tahmin)
- Bu tahmin gerçek mask ile karşılaştırılıyor (BCE Loss)
- Hata geriye yayılıp sadece LoRA parametreleri güncelleniyor

**Mixed Precision (AMP) nedir?**
GPU belleğinden tasarruf sağlayan bir teknik. Bazı hesaplamaları 16-bit ile yaparak hem hızlanır hem de daha az bellek kullanılır. T4 GPU'da bu önemli.

**BCE Loss nedir?**
Binary Cross Entropy — her piksel için "hasar var mı yok mu?" sorusunu ne kadar doğru cevapladığını ölçer. Loss düştükçe model daha iyi tahmin yapıyor demektir.

In [ ]:
import torch.nn as nn
from tqdm import tqdm

# --- IoU hesaplama fonksiyonu ---
# IoU (Intersection over Union) = kesişim / birleşim
# Segmentation'da en önemli metrik — loss'tan daha güvenilir
def hesapla_iou(pred_logits, gt_mask, threshold=0.5):
    """
    Tahmin ve gerçek mask arasındaki IoU'yu hesaplar.

    Args:
        pred_logits: model çıktısı (sigmoid ÖNCESİ ham değerler)
        gt_mask: gerçek mask (0/1)
        threshold: sigmoid sonrası bu değerin üstü = hasar var

    Returns:
        float: IoU skoru (0.0 = hiç örtüşme yok, 1.0 = mükemmel)
    """
    pred_binary = (pred_logits.sigmoid() > threshold).float()
    intersection = (pred_binary * gt_mask).sum()
    union = pred_binary.sum() + gt_mask.sum() - intersection

    if union == 0:
        # İkisi de boşsa (hasar yok) → mükemmel eşleşme
        return 1.0
    return (intersection / union).item()


# --- Kayıp fonksiyonu ve optimizer ---
loss_fn   = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
)

NUM_EPOCHS = 3  # Test için 3 epoch, başarılıysa artırılabilir

# Mixed precision — GPU belleğinden tasarruf sağlar
scaler = torch.amp.GradScaler("cuda")

print(f"Eğitim başlıyor: {NUM_EPOCHS} epoch, {len(train_loader)} batch/epoch")
print(f"Text prompt: '{TEXT_PROMPT}'")
print(f"Mask boyutu: {MASK_OUTPUT_SIZE}x{MASK_OUTPUT_SIZE}")
print("=" * 60)

for epoch in range(NUM_EPOCHS):
    # =====================
    #       EĞİTİM
    # =====================
    model.train()
    train_loss = 0.0
    train_iou  = 0.0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [train]"):
        pixel_values   = batch["pixel_values"].to(DEVICE)
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        gt_mask        = batch["ground_truth_mask"].to(DEVICE)  # (B, 288, 288)

        # Forward pass — TAM pipeline (encoder → DETR → mask decoder)
        with torch.amp.autocast("cuda"):
            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
            )

            # semantic_seg: (B, 1, 288, 288) → (B, 288, 288)
            pred_mask = outputs.semantic_seg.squeeze(1)

            # Loss: her piksel için tahmin vs gerçek
            loss = loss_fn(pred_mask, gt_mask)

        # Geri yayılım (mixed precision ile)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        train_iou  += hesapla_iou(pred_mask.detach(), gt_mask)

    avg_train_loss = train_loss / len(train_loader)
    avg_train_iou  = train_iou  / len(train_loader)

    # =====================
    #     VALIDATION
    # =====================
    model.eval()
    val_loss = 0.0
    val_iou  = 0.0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [val]"):
            pixel_values   = batch["pixel_values"].to(DEVICE)
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            gt_mask        = batch["ground_truth_mask"].to(DEVICE)

            with torch.amp.autocast("cuda"):
                outputs = model(
                    pixel_values=pixel_values,
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                )
                pred_mask = outputs.semantic_seg.squeeze(1)
                loss = loss_fn(pred_mask, gt_mask)

            val_loss += loss.item()
            val_iou  += hesapla_iou(pred_mask, gt_mask)

    avg_val_loss = val_loss / len(val_loader)
    avg_val_iou  = val_iou  / len(val_loader)

    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print(f"  Train → Loss: {avg_train_loss:.4f} | IoU: {avg_train_iou:.4f}")
    print(f"  Val   → Loss: {avg_val_loss:.4f} | IoU: {avg_val_iou:.4f}")

    # Checkpoint kaydet — sadece LoRA adapter ağırlıklarını kaydet
    # (base model zaten HuggingFace'ten yüklenebilir, sadece öğrenilen kısım yeter)
    adapter_path = f"{CHECKPOINT_DIR}/epoch_{epoch+1}_lora"
    model.save_pretrained(adapter_path)
    print(f"  LoRA adapter kaydedildi: {adapter_path}")

print("\n" + "=" * 60)
print("Eğitim tamamlandı!")

## Adım 8 — Görselleştirme

Eğitim bitti, ama sayılara bakmak yetmez.
Modelin **gerçekten ne tahmin ettiğini** gözle görmeliyiz.

Her örnek için 4 panel gösteriyoruz:
1. **Orijinal görsel** — köprü fotoğrafı
2. **Gerçek mask** — annotation'dan üretilen hasar bölgesi (beyaz = hasar)
3. **Tahmin mask** — modelin tahmini (beyaz = hasar diyor)
4. **Overlay** — tahmin, orijinal görselin üzerine bindirilmiş (kırmızı = hasar)

In [ ]:
import matplotlib.pyplot as plt

# Validation setinden 4 örnek göster
NUM_EXAMPLES = 4

model.eval()
fig, axes = plt.subplots(NUM_EXAMPLES, 4, figsize=(16, 4 * NUM_EXAMPLES))

# Tek örnek varsa axes'i 2D yap
if NUM_EXAMPLES == 1:
    axes = axes.reshape(1, -1)

sample_iter = iter(val_loader)
shown = 0

with torch.no_grad():
    for batch in sample_iter:
        if shown >= NUM_EXAMPLES:
            break

        pixel_values   = batch["pixel_values"].to(DEVICE)
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        gt_mask        = batch["ground_truth_mask"]  # CPU'da kalsın, görselleştirme için

        with torch.amp.autocast("cuda"):
            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
            )
        pred_logits = outputs.semantic_seg.squeeze(1).cpu()  # (B, 288, 288)

        # Batch'teki her örnek için
        for i in range(pixel_values.shape[0]):
            if shown >= NUM_EXAMPLES:
                break

            # Orijinal görseli processor'dan geri al (yaklaşık — normalize geri alınıyor)
            img_tensor = batch["pixel_values"][i]  # (3, 1008, 1008)
            # ImageNet normalizasyonunu geri al
            mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
            std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
            img_display = (img_tensor * std + mean).clamp(0, 1)
            img_display = img_display.permute(1, 2, 0).numpy()  # (H, W, 3)

            gt   = gt_mask[i].numpy()                           # (288, 288)
            pred = (pred_logits[i].sigmoid() > 0.5).float().numpy()  # (288, 288)
            iou  = hesapla_iou(pred_logits[i].unsqueeze(0), gt_mask[i].unsqueeze(0))

            # --- 4 panel ---
            # 1) Orijinal görsel
            axes[shown, 0].imshow(img_display)
            axes[shown, 0].set_title("Orijinal")
            axes[shown, 0].axis("off")

            # 2) Gerçek mask
            axes[shown, 1].imshow(gt, cmap="gray", vmin=0, vmax=1)
            axes[shown, 1].set_title("Gerçek Mask")
            axes[shown, 1].axis("off")

            # 3) Tahmin mask
            axes[shown, 2].imshow(pred, cmap="gray", vmin=0, vmax=1)
            axes[shown, 2].set_title(f"Tahmin (IoU: {iou:.3f})")
            axes[shown, 2].axis("off")

            # 4) Overlay — tahmini kırmızı olarak orijinalin üzerine koy
            # Görseli mask boyutuna küçült
            img_small = torch.nn.functional.interpolate(
                img_tensor.unsqueeze(0), size=(288, 288), mode="bilinear", align_corners=False
            ).squeeze(0)
            img_small = (img_small * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
            overlay = img_small.copy()
            # Tahmin edilen hasar bölgelerini kırmızıya boya (yarı saydam)
            overlay[pred == 1, 0] = 1.0   # kırmızı kanal
            overlay[pred == 1, 1] *= 0.3  # yeşili azalt
            overlay[pred == 1, 2] *= 0.3  # maviyi azalt

            axes[shown, 3].imshow(overlay)
            axes[shown, 3].set_title("Overlay (kırmızı=tahmin)")
            axes[shown, 3].axis("off")

            shown += 1

plt.tight_layout()
plt.savefig(f"{CHECKPOINT_DIR}/gorsel_sonuclar.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Görselleştirme kaydedildi: {CHECKPOINT_DIR}/gorsel_sonuclar.png")

## Özet

Bu notebook tamamlandığında:
- Model **gerçek pipeline** ile eğitildi: Image + Text → Encoder → DETR → Mask Decoder
- `semantic_seg` çıktısı kullanılarak **piksel piksel** segmentation öğrenildi
- Her epoch'ta **IoU metriki** ile model kalitesi ölçüldü (loss'a güvenmekten daha güvenilir)
- Her epoch sonunda **LoRA adapter** checkpoint'u Drive'a kaydedildi
- Görselleştirme ile modelin gerçekten ne tahmin ettiği **gözle** kontrol edildi

**Sonuçları değerlendirme:**
- IoU yükseliyorsa → model gerçekten hasar bölgelerini öğreniyor ✔
- IoU 0'a yakınsa → model henüz anlamlı bir şey üretemiyor
- Train IoU yüksek ama val IoU düşükse → overfitting var
- Görselleştirmede tahminler gerçek mask'a benziyorsa → her şey yolunda

**Sonraki adımlar:**
- Test başarılıysa `TEST_SAMPLE_SIZE = 100` yerine `max_samples=None` ile tam veri setiyle çalıştır
- Faz 5'te inference ve demo yapılacak